# 2 · Build a dataset

Start with ten labels from notebook 0. Find a Sentinel-2 image for each label's place and day, then save the pairs:

```text
data/my_dataset/
├── train/  sentinel-2-l2a/<tile>.tif  label/<tile>.tif
├── val/    ...
├── test/   ...
└── catalog.parquet
```

We will also save six forest scenes for prediction in notebook 3. This notebook needs internet access.


**Data and interpretation.** These are expert-annotated training labels from [Tait et al. (2021)](https://doi.org/10.1594/PANGAEA.933475), paired with Sentinel-2 L2A imagery in the [teaching subset](https://huggingface.co/datasets/fatmur/dynamicworld-300). They have ten classes, including cloud, and nodata 255. They are not the automated nine-class Dynamic World probability product.

Keep tiles in their original train/validation/test splits before making chips. Nearby or overlapping tiles can still leak spatial information: inspect footprints before claiming geographic generalization. These short runs are workflow checks, not evidence of mapping accuracy.


**Run:** use the repository `.venv` kernel with `examples/` as the working directory. Requires `uv sync`, internet access, and the data paths described below.

In [ ]:
from pathlib import Path

from dask.diagnostics.progress import ProgressBar
from huggingface_hub import snapshot_download

import xarray as xr

import geosave as gs

# Show progress when reading pixels.
pbar = ProgressBar(minimum=1)
pbar.register()

root = Path(
    snapshot_download(
        "fatmur/dynamicworld-300",
        repo_type="dataset",
        revision="5e65026ad47e1f02f9a48a3efee1028db3bb0d1a",
        local_dir="data/dynamic_world",
    )
)

## Read a label's place and date

Its `region` describes geometry and time for searching. Its `geobox` defines the exact pixel grid for loading.


In [ ]:
label_path = sorted((root / "train/labels").glob("*.tif"))[0]
label = gs.open_raster(label_path)

region = label.gs.region
grid = label.gs.geobox
place = region.format("{lat:.2f}, {lon:.2f} · {start:%Y-%m-%d} · {width:g} m across")
f"{place} at {abs(grid.resolution.x):g} m"

In [ ]:
label.gs.plot(xlabel="")

## Find the image

A [STAC](https://stacspec.org) catalog lists scenes by place and date.

Choose four Sentinel-2 bands. Request cloud-cover ordering and keep one scene. Service ordering can vary; inspect the returned cloud cover when choosing production imagery. Search the label's own day so the image matches the label.


In [ ]:
BANDS = ["B02", "B03", "B04", "B08"]

client = gs.StacClient.open("planetary_computer")
collection = client.collection("sentinel-2-l2a")
query = gs.StacQuery(collections=["sentinel-2-l2a"], max_items=1).sort_by(
    "properties.eo:cloud_cover"
)
options: gs.stac.StacLoadOptions = dict(bands=BANDS, dtype="uint16", nodata=0)

Search the full day without loading pixels.


In [ ]:
day = region.replace(timespan=region.format("{start:%Y-%m-%d}"))
search = query.set_region(day)
selected = client.search_items(search)
for item in selected:
    print(item.id, item.properties.get("eo:cloud_cover"))

Load the image on the label's grid. Record its reflectance scale and RGB bands. Plotting reads the pixels.


In [ ]:
image = gs.stac.load_dataset(
    selected, geobox=grid, collection=collection, patch_url=client.patch_url, **options
)
image = image.gs.write_scale_offset(1e-4, 0)
image = image.gs.write_rgb(red="B04", green="B03", blue="B02")

pair = xr.DataTree.from_dict({"sentinel-2-l2a": image, "label": label})
pair.gs.plot(cols=2, xlabel="")

## Save and catalog the pair

`to_catalog` writes each group as a Cloud-Optimized GeoTIFF and returns a catalog of what it wrote: a GeoDataFrame with one STAC Item per group and date. `{collection}` gives each group its own folder; both use the label's filename.

A catalog stores file paths and metadata, not pixels. Add `split` as an ordinary column.

In [ ]:
import pystac

folder = Path("data/my_dataset")

label_collection = pystac.Collection(
    id="label",
    description="Expert-annotated Dynamic World training labels",
    extent=collection.extent,
)
image_catalog = image.gs.to_catalog(
    folder / "train",
    collection=collection,
    split_asset=(),
    format_paths=f"sentinel-2-l2a/{label_path.stem}.tif",
    id=label_path.stem,
    overwrite=True,
)
label_catalog = label.gs.to_catalog(
    folder / "train",
    collection=label_collection,
    split_asset=(),
    format_paths=f"label/{label_path.stem}.tif",
    id=label_path.stem,
    overwrite=True,
)
items = gs.GeoCatalog.concat([image_catalog, label_catalog])
items["split"] = "train"
items[["id", "collection", "datetime", "split"]]

## Build all three splits

`fetch_image` loads an image. `save_tile` saves and catalogs one image/label pair. `build_dataset` repeats it and returns the catalog of one split. Each call receives its catalog client, collection, and query explicitly. `BANDS` is the fixed band selection for this project; the query controls which scene is chosen.

The dates in this lesson are 2019/2020, before the Sentinel-2 offset change. The helper explicitly records their reflectance scale (`1e-4`) and offset (`0`). When adapting to another product or processing baseline, verify its scale/offset before reusing these values; an incorrect offset changes spectral indices and model inputs.

In [ ]:
from tqdm.auto import tqdm


def fetch_image(region, grid, *, client, collection, query):
    search = query.set_region(region)
    selected = client.search_items(search)

    image = gs.stac.load_dataset(
        selected,
        geobox=grid,
        collection=collection,
        patch_url=client.patch_url,
        bands=BANDS,
        dtype="uint16",
        nodata=0,
    )
    # The selected 2019/2020 Planetary Computer scenes use scale 1e-4, offset 0.
    image = image.gs.write_scale_offset(1e-4, 0)

    return image.gs.write_rgb(red="B04", green="B03", blue="B02")

Use the label's place, day, and grid to save one matching pair.

In [ ]:
def save_tile(label_path, folder, *, client, collection, query):
    import pystac

    label = gs.open_raster(label_path)
    region = label.gs.region

    day = region.replace(timespan=region.format("{start:%Y-%m-%d}"))
    image = fetch_image(
        day, label.gs.geobox, client=client, collection=collection, query=query
    )

    label_collection = pystac.Collection(
        id="label",
        description="Expert-annotated Dynamic World training labels",
        extent=collection.extent,
    )
    image_catalog = image.gs.to_catalog(
        folder,
        collection=collection,
        split_asset=(),
        format_paths=f"sentinel-2-l2a/{label_path.stem}.tif",
        id=label_path.stem,
        overwrite=True,
    )
    label_catalog = label.gs.to_catalog(
        folder,
        collection=label_collection,
        split_asset=(),
        format_paths=f"label/{label_path.stem}.tif",
        id=label_path.stem,
        overwrite=True,
    )
    return gs.GeoCatalog.concat([image_catalog, label_catalog])


Repeat that operation for one split, then combine its catalog rows.

In [ ]:
def build_dataset(labels, folder, split, *, client, collection, query):
    catalogs = []
    for path in tqdm(labels, desc=split):
        catalog = save_tile(
            path, folder / split, client=client, collection=collection, query=query
        )
        catalogs.append(catalog)

    catalog = gs.GeoCatalog.concat(catalogs)
    catalog["split"] = split

    return catalog

Use six training labels, two validation labels and two test labels for this short example.


In [ ]:
pbar.unregister()

train_labels = sorted((root / "train/labels").glob("*.tif"))[:6]
val_labels = sorted((root / "val/labels").glob("*.tif"))[:2]
test_labels = sorted((root / "test/labels").glob("*.tif"))[:2]

train = build_dataset(
    train_labels, folder, "train", client=client, collection=collection, query=query
)
val = build_dataset(
    val_labels, folder, "val", client=client, collection=collection, query=query
)
test = build_dataset(
    test_labels, folder, "test", client=client, collection=collection, query=query
)

catalog = gs.GeoCatalog.concat([train, val, test])
catalog.gs.to_geoparquet(folder / "catalog.parquet", overwrite=True)
catalog

Open the catalog on a map. Select rows, then use **Open selected items** to view their rasters.

The server runs in the background, so you can keep working in the next cells. Running this cell again replaces the explorer on the same port.

Open the local URL printed by the server. `open=False` prevents an automatic browser launch.

### Optional: open the catalog viewer

Run this separately when you want a local viewer. Stop it after browsing.

```python
explorer = catalog.gs.explore()
server = explorer.serve(threaded=True, open=False)
# When finished:
explorer.stop()
```

## Reopen the catalog

The saved GeoParquet works as a STAC source, searched like Planetary Computer.

In [ ]:
saved = gs.StacClient.open(folder / "catalog.parquet")
saved.search(gs.StacQuery(collections=["label"]))

## Pair images with labels

`sample` groups catalog rows into samples without opening a file. `collections` selects imagery and labels within each daily window (`freq="D"`). A label is dated at midnight and its image at capture time. Inspect the resulting rows to confirm that each sample has both collections before opening it.

In [ ]:
samples = catalog.gs.sample(collections=["sentinel-2-l2a", "label"], freq="D")
samples[0]

Each sample opens as a lazy DataTree with collection parents and grid leaves, `/{collection}/{res}`. Notebook 3 opens the image and label from each sample, prepares them, and cuts matching model-sized chips.


In [ ]:
samples[0].gs.to_datatree()

## Start from a shape

Open the forest outline in `khdtk.geojson` and give it a 10 m grid. Fetch one scene from each month for prediction.


In [ ]:
from odc.geo.geobox import GeoBox

region = gs.open_vector("khdtk.geojson")
forest_region = gs.GeoRegion.from_geometry(region)
# KHDTK is in UTM zone 49S. Reuse this grid for every month's image.
region_grid = GeoBox.from_geopolygon(
    forest_region.extent, crs="EPSG:32749", resolution=10
)

months = ["2019-05", "2019-06", "2019-07", "2019-08", "2019-09", "2019-10"]
pictures = []
for month in months:
    month_region = forest_region.replace(timespan=month)
    picture = fetch_image(
        month_region, region_grid, client=client, collection=collection, query=query
    )
    pictures.append(picture)

season = xr.concat(
    pictures, dim="time", join="exact", combine_attrs=gs.attrs.combine_attrs
).compute()
season

Save and catalog the six scenes for notebook 3. Each file keeps its acquisition date; these are individual scenes, not monthly composites.

In [ ]:
khdtk = Path("data/khdtk")

season_catalog = season.gs.to_catalog(
    khdtk,
    format_paths="{collection}/{time:%Y%m%d}.tif",
    collection=collection,
    split_asset=(),
    format_id="{time:%Y%m%d}",
    overwrite=True,
)
season_catalog.gs.to_geoparquet(khdtk / "catalog.parquet", overwrite=True)
season_catalog[["id", "collection", "datetime"]]

## What happened

We saved image/label pairs in fixed splits and six forest scenes for prediction. Notebook 3 uses their catalogs to make model inputs.


## More things you can do with this dataset (optional)

The dataset and forest observations are saved. Continue to [notebook 3](03_inputs.ipynb), or try one of these extensions.

### Keep a shortlist when a label's scene is cloudy
Look at several candidates before downloading pixels. Keep the label's acquisition day fixed here; changing dates may change what the label describes.

In [ ]:
candidates = gs.StacQuery(collections=["sentinel-2-l2a"], max_items=5).sort_by(
    "properties.eo:cloud_cover"
)
client.search(candidates.set_region(day))

### Expand the forest monitoring period
Add another month to `months`, reuse `region_grid`, and inspect image quality before predicting it. Keep a record of missing observations rather than silently replacing them with another season.

### Work with a boundary your team recognizes
Replace `khdtk.geojson` with your project boundary and choose an appropriate projected grid. Keep the downloaded imagery and catalog together so another person can reopen the same observations.

**Next:** [train on these pairs and map the forest](03_inputs.ipynb). The new forest scenes have no reference labels, so they are for application and visual review, not accuracy measurement.

### Compare greenness with the predicted land cover

When reviewing a predicted change in tree cover, compare it with NDVI from the same observation. GeoSave's NDVI function uses red and near-infrared bands. This adds a review layer without changing the four-band model inputs.

In [ ]:
ndvi = gs.features.ndvi(season, nir="B08", red="B04")
with_ndvi = season.assign(ndvi=ndvi)
with_ndvi

In [ ]:
with_ndvi.gs.explore("ndvi", basemap="Imagery", cmap="RdYlGn") * region